# Ein- und Austritte – 1) Daten laden und aufbereiten
Führt die 4 SQL-Dateien aus, berechnet Bestand per 1.1., Wochen- und Tagesverlauf und speichert
die Resultate im Ordner `Resultate/`. Die Grafiken macht das Notebook `ein_austritte_grafiken`.

Speicher: Die Aufbereitung ist Python (Snowpark), wird aber in Snowflake gerechnet (Warehouse).
Ins Notebook geladen werden nur die kleinen Resultate (Bestand, Wochen, Tage), nie die Rohdaten.

In [ ]:
import datetime as dt
import re
from pathlib import Path

import pandas as pd
from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session

try:
    from IPython.display import display
except ImportError:
    display = print

session = get_active_session()

AKT_JAHR = 2027
ERSTES_JAHR = 2022
JAHRE = list(range(ERSTES_JAHR, AKT_JAHR + 1))
# Bestand zusätzlich per 1.1. des Folgejahres (bereits erfasste Ein-/Austritte per 1.1.2028)
BESTAND_JAHRE = JAHRE + [AKT_JAHR + 1]
date = dt.date.today()

# Ordner mit den .sql-Dateien, relativ zum Notebook (inkl. Unterordner);
# Gross-/Kleinschreibung, Leerzeichen, "-" und "_" werden beim Suchen ignoriert
SQL_ORDNER = "../Ein-Austritte"
SQL_FILES = {
    "okp_eintritte": "OKP-Eintritte2.sql",
    "vvg_eintritte": "VVG-Eintritte2.sql",
    "okp_austritte": "OKP-Austritte2.sql",
    "vvg_austritte": "VVG-Austritte2.sql",
}
# Ausgabeordner relativ zum Notebook (das Notebook liegt bereits in "Resultate")
RESULTAT_ORDNER = Path(".")


# Zwischenresultat als temporäre Tabelle speichern, damit die 4 Abfragen nur einmal laufen
# (braucht Recht auf temporäre Tabellen; sonst automatisch ohne)
ZWISCHENSPEICHERN = True


def sql_datei_finden(dateiname):
    norm = lambda n: re.sub(r"[\s_\-]", "", n.lower())
    kandidaten = list(Path(SQL_ORDNER).rglob("*.sql"))
    for pfad in kandidaten:
        if norm(pfad.name) == norm(dateiname):
            return pfad
    raise FileNotFoundError(f"{dateiname} nicht gefunden in {Path(SQL_ORDNER).resolve()}. "
                            f"Gefundene .sql-Dateien: {[str(k) for k in kandidaten]}")


def abfrage(name):
    """Snowpark-DataFrame der SQL-Datei (Befehle vor dem letzten, z.B. USE ROLE, werden nur ausgeführt).
    Es werden noch keine Daten geladen."""
    pfad = sql_datei_finden(SQL_FILES[name])
    print(f"{name}: {pfad}")
    sql = pfad.read_text(encoding="utf-8")
    sql = re.sub(r"/\*.*?\*/", "", sql, flags=re.S)        # Kommentare entfernen (enthalten evtl. ";")
    sql = re.sub(r"--[^\n]*", "", sql)
    befehle = [b.strip() for b in sql.split(";") if b.strip()]
    for befehl in befehle[:-1]:
        session.sql(befehl).collect()
    return session.sql(befehle[-1])


def kw_ch(d):
    """ISO-KW mit Korrektur am Jahreswechsel (wie R): KW 1/53 im Dezember -> 52, KW 52/53 im Januar -> 1."""
    iso = F.call_function("WEEKISO", d)
    return (F.when((F.month(d) == 12) & iso.isin(1, 53), F.lit(52))
             .when((F.month(d) == 1) & (iso >= 52), F.lit(1))
             .otherwise(iso))


def kw_ch_pd(d):
    """Gleiche Regel für pandas (für die aufgefüllten Tage und die aktuelle KW)."""
    iso = d.dt.isocalendar().week.astype("Int64")
    m = d.dt.month
    kw = iso.copy()
    kw[((m == 12) & iso.isin([1, 53])).fillna(False).astype(bool)] = 52
    kw[((m == 1) & (iso >= 52)).fillna(False).astype(bool)] = 1
    return kw


def zwischenspeichern(df):
    if not ZWISCHENSPEICHERN:
        return df
    try:
        return df.cache_result()
    except Exception as e:
        print(f"Zwischenspeichern nicht möglich, rechne ohne: {e}")
        return df


def zu_pandas(df):
    pdf = df.to_pandas()
    for c in ("DATUM", "STICHTAG"):
        if c in pdf.columns:
            pdf[c] = pd.to_datetime(pdf[c])
    return pdf


AKT_KW = int(kw_ch_pd(pd.Series([pd.Timestamp(date)])).iloc[0])

In [ ]:
# 1) Abfragen: ungefilterter Output pro SQL-Datei (wird in Schritt 6 exportiert).
#    Zwischengespeichert, damit jede Abfrage nur einmal läuft (Aufbereitung und Export nutzen dasselbe Resultat).
abfragen = {name: zwischenspeichern(abfrage(name)) for name in SQL_FILES}


# Nur die benötigten Spalten, Datumsspalten als DATE (alles in Snowflake)
def basis(name, flag, mutation, vertrag):
    return (abfragen[name]
            .select(F.col("PARTNERNR"), F.col("SPARTE"), F.col(flag).alias("FLAG"),
                    F.to_date(F.col("STATEFROM")).alias("STATEFROM"),
                    F.to_date(F.col("STATEUPTO")).alias("STATEUPTO"),
                    F.to_date(F.col(mutation)).alias("MUTATIONSDATUM"),
                    F.to_date(F.col(vertrag)).alias("VERTRAGSDATUM")))


# Eintritte: MUTATIONSDATUM = ABSCHLUSSDATUM, VERTRAGSDATUM = EINTRITTSDATUM_SPARTE
# Austritte: MUTATIONSDATUM = MUTATIONSDATUM_*_AUSTRITT, VERTRAGSDATUM = AUSTRITTSDATUM_SPARTE
roh_ein = (basis("okp_eintritte", "IS_EINTRITT_SPARTE", "ABSCHLUSSDATUM", "EINTRITTSDATUM_SPARTE")
           .union_all(basis("vvg_eintritte", "IS_EINTRITT_SPARTE", "ABSCHLUSSDATUM", "EINTRITTSDATUM_SPARTE")))
roh_aus = (basis("okp_austritte", "IS_AUSTRITT_SPARTE", "MUTATIONSDATUM_OKP_AUSTRITT", "AUSTRITTSDATUM_SPARTE")
           .union_all(basis("vvg_austritte", "IS_AUSTRITT_SPARTE", "MUTATIONSDATUM_PRODUKT_AUSTRITT", "AUSTRITTSDATUM_SPARTE")))

In [ ]:
# 2) Pro Person, Sparte und Vertragsdatum eine Zeile (frühester Abschluss / letzte Mutation)
eintritte_kw = (roh_ein.filter(F.col("FLAG") == 1)
                .group_by("PARTNERNR", "SPARTE", "VERTRAGSDATUM")
                .agg(F.min("MUTATIONSDATUM").alias("MUTATIONSDATUM"), F.min("STATEFROM").alias("STATEFROM")))
austritte_kw = (roh_aus.filter(F.col("FLAG") == 1)
                .group_by("PARTNERNR", "SPARTE", "VERTRAGSDATUM")
                .agg(F.max("MUTATIONSDATUM").alias("MUTATIONSDATUM"), F.max("STATEUPTO").alias("STATEUPTO")))

# Ein- und Austritte in einer Tabelle (wie R): DATUM = ABSCHLUSSDATUM (Eintritt) bzw. MUTATIONSDATUM (Austritt)
bewegungen = (eintritte_kw.select("PARTNERNR", "SPARTE", F.col("MUTATIONSDATUM").alias("DATUM"),
                                  F.lit("Eintritt").alias("ART"))
              .union_all(austritte_kw.select("PARTNERNR", "SPARTE", F.col("MUTATIONSDATUM").alias("DATUM"),
                                             F.lit("Austritt").alias("ART")))
              .filter(F.year("DATUM").between(ERSTES_JAHR, AKT_JAHR))
              .with_column("JAHR", F.year("DATUM"))
              .with_column("KW", kw_ch(F.col("DATUM"))))


def zaehlen(df, gruppen):
    """Distinct PARTNERNR pro Gruppe und ART (in Snowflake), als Spalten EINTRITTE / AUSTRITTE."""
    n = zu_pandas(df.group_by(*gruppen, "ART").agg(F.count_distinct("PARTNERNR").alias("N")))
    return (n.pivot_table(index=gruppen, columns="ART", values="N", fill_value=0, aggfunc="sum")
             .reindex(columns=["Eintritt", "Austritt"], fill_value=0)
             .rename(columns={"Eintritt": "EINTRITTE", "Austritt": "AUSTRITTE"})
             .rename_axis(columns=None))

In [ ]:
# 3) Bestand per Stichtag 1.1. pro Sparte und Jahr (distinct PARTNERNR), inkl. 1.1. des Folgejahres
#    BESTAND:            Deckung läuft am Stichtag (STATEFROM <= Stichtag <= STATEUPTO, leeres STATEUPTO = offen)
#    EIN_RUECKW_ERFASST: im Bestand, Abschluss aber erst nach dem Stichtag erfasst
#    AUS_RUECKW_ERFASST: Deckung schon beendet, Austritt aber erst nach dem Stichtag erfasst
#    GEKUENDIGT:         Kündigung erfasst, Deckung läuft am Stichtag noch
stichtage = session.create_dataframe([[j, dt.date(j, 1, 1)] for j in BESTAND_JAHRE], schema=["JAHR", "STICHTAG"])
st = F.col("STICHTAG")


def fall_zaehlen(df, fall, bedingung):
    return (df.cross_join(stichtage)
              .filter(bedingung)
              .group_by("SPARTE", "JAHR", "STICHTAG")
              .agg(F.count_distinct("PARTNERNR").alias("N"))
              .with_column("FALL", F.lit(fall)))


zeitraeume = (roh_ein.select("PARTNERNR", "SPARTE", "STATEFROM", "STATEUPTO")
              .union_all(roh_aus.select("PARTNERNR", "SPARTE", "STATEFROM", "STATEUPTO"))
              .distinct())
faelle = (fall_zaehlen(zeitraeume, "BESTAND",
                       (F.col("STATEFROM") <= st) & (F.col("STATEUPTO").is_null() | (F.col("STATEUPTO") >= st)))
          .union_all(fall_zaehlen(eintritte_kw, "EIN_RUECKW_ERFASST",
                                  (F.col("VERTRAGSDATUM") <= st) & (F.col("MUTATIONSDATUM") > st)))
          .union_all(fall_zaehlen(austritte_kw, "AUS_RUECKW_ERFASST",
                                  (F.col("VERTRAGSDATUM") < st) & (F.col("MUTATIONSDATUM") >= st)))
          .union_all(fall_zaehlen(austritte_kw, "GEKUENDIGT",
                                  (F.col("MUTATIONSDATUM") <= st) & (F.col("VERTRAGSDATUM") >= st))))

bestand = (zu_pandas(faelle)
           .pivot_table(index=["SPARTE", "JAHR", "STICHTAG"], columns="FALL", values="N",
                        fill_value=0, aggfunc="sum")
           .reindex(columns=["BESTAND", "EIN_RUECKW_ERFASST", "AUS_RUECKW_ERFASST", "GEKUENDIGT"], fill_value=0)
           .rename_axis(columns=None)
           .reset_index())
bestand = bestand[bestand["BESTAND"] > 0].sort_values(["SPARTE", "JAHR"]).reset_index(drop=True)
display(bestand)

In [ ]:
# 4) Wochenverlauf: Personen pro Sparte, Jahr, KW; alle KW 1–52 (aktuelles Jahr bis aktuelle KW), fehlend = 0
kw = zaehlen(bewegungen, ["SPARTE", "JAHR", "KW"])
sparten = sorted(kw.index.get_level_values("SPARTE").dropna().unique())
voll = pd.MultiIndex.from_product([sparten, JAHRE, range(1, 53)], names=["SPARTE", "JAHR", "KW"])
kw = kw.reindex(voll, fill_value=0).reset_index()
kw = kw[(kw["JAHR"] < AKT_JAHR) | (kw["KW"] <= AKT_KW)].reset_index(drop=True)
kw["VERHAELTNIS"] = kw["AUSTRITTE"] / kw["EINTRITTE"]
kw["NETTO"] = kw["EINTRITTE"] - kw["AUSTRITTE"]
display(kw.tail(10))

In [ ]:
# 5) Tagesverlauf: Personen pro Sparte und Tag, jeder Tag ab 1.1.ERSTES_JAHR bis heute, fehlend = 0
WOCHENTAGE = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]
tag = zaehlen(bewegungen.filter(F.col("DATUM") <= F.lit(date)), ["SPARTE", "DATUM"])
voll = pd.MultiIndex.from_product([sparten,
                                   pd.date_range(pd.Timestamp(ERSTES_JAHR, 1, 1), pd.Timestamp(date), freq="D")],
                                  names=["SPARTE", "DATUM"])
tag = tag.reindex(voll, fill_value=0).reset_index()
tag.insert(1, "JAHR", tag["DATUM"].dt.year)
tag.insert(2, "KW", kw_ch_pd(tag["DATUM"]))
tag.insert(4, "WOCHENTAG", tag["DATUM"].dt.dayofweek.map(dict(enumerate(WOCHENTAGE))))
display(tag.tail(10))



In [ ]:
# 6) Resultate speichern (für das Grafik-Notebook und als Export, wie bisher write.csv2 in R)
RESULTAT_ORDNER.mkdir(exist_ok=True)
for name, df in {"bestand": bestand, "kw": kw, "tag": tag}.items():
    df.to_csv(RESULTAT_ORDNER / f"{name}_{date}.csv", sep=";", decimal=",", index=False)
pers = zu_pandas(bewegungen.select("PARTNERNR", "SPARTE", "ART", "DATUM", "JAHR"))
pers.to_csv(RESULTAT_ORDNER / f"bewegungen_{date}.csv", sep=";", index=False)

# Ungefilterter Output der 4 SQL-Abfragen als CSV, plus die ausgeführte SQL-Datei als Kopie.
# Die Rohdaten werden in Portionen geschrieben, damit nie alles gleichzeitig im Notebook liegt.
EXPORT_ABFRAGEN = True
if EXPORT_ABFRAGEN:
    for name, df in abfragen.items():
        ziel = RESULTAT_ORDNER / f"abfrage_{name}_{date}.csv"
        zeilen = 0
        with open(ziel, "w", encoding="utf-8", newline="") as f:
            for i, teil in enumerate(df.to_pandas_batches()):
                teil.to_csv(f, sep=";", decimal=",", index=False, header=(i == 0))
                zeilen += len(teil)
            if zeilen == 0:                      # leeres Resultat: wenigstens die Spaltennamen schreiben
                f.write(";".join(df.columns) + "\n")
        sql_kopie = RESULTAT_ORDNER / f"abfrage_{name}_{date}.sql"
        sql_kopie.write_text(sql_datei_finden(SQL_FILES[name]).read_text(encoding="utf-8"), encoding="utf-8")
        print(f"{name}: {zeilen} Zeilen -> {ziel.name} (SQL: {sql_kopie.name})")
print(f"gespeichert in {RESULTAT_ORDNER.resolve()}")

# Optional zusätzlich als Tabellen in Snowflake (Datenbank/Schema mit Schreibrechten eintragen)
EXPORT_TABELLEN = False
if EXPORT_TABELLEN:
    for name, df in {"EA_BESTAND_STICHTAG": bestand, "EA_PRO_KW": kw, "EA_PRO_TAG": tag}.items():
        session.write_pandas(df, "<DB>.<SCHEMA>." + name, auto_create_table=True, overwrite=True)
    # Rohdaten der Abfragen direkt in Snowflake speichern (ohne Umweg über das Notebook)
    for name, df in abfragen.items():
        df.write.save_as_table("<DB>.<SCHEMA>.EA_ABFRAGE_" + name.upper(), mode="overwrite")